# 1102 42일차

## 1. Hold-out 분할의 한계

전체 데이터를 고정된 비율의 train, val, test로 한 번만 나누는 방식

데이터 크기가 작을 때 테스트로 떼어둔 데이터를 훈련에 쓰지 못해 데이터 손실이 생기고, 어떤 데이터가 테스트셋에 들어갔느냐에 따라 평가 점수가 크게 흔들림

1. `train` : 모델 가중치를 학습시키는 데이터 (보통 60~80%)
2. `val` : 에포크마다 과적합을 모니터링하고 하이퍼파라미터를 튜닝하는 검증 데이터 (보통 10~20%)
3. `test` : 학습에 전혀 쓰이지 않고 최종 모델의 성능만 측정한 뒤 버려지는 평가 데이터 (보통 10~20%)

- test 세트에 우연히 쉬운 데이터나 이상치가 몰리면 성능이 왜곡되어 모델의 실제 일반화 성능을 믿기 어려움
- 데이터가 적을 때 test 세트를 크게 잡으면 train 데이터가 부족해져 모델 성능 자체가 떨어짐
- 한 번 평가하고 버려지는 test 데이터도 학습용으로 쓸 수 있으면 데이터 낭비를 줄일 수 있음

## 2. K-Fold 교차 검증

전체 데이터를 k개 묶음(Fold)으로 나누고, 매번 다른 묶음을 테스트셋으로 바꿔가며 k번 학습·평가해 평균을 내는 검증 방법 (`m02_kfold_00_iris`, `m02_kfold_01_california`)

고정된 test 세트 하나로 평가할 때 생기는 편향을 없애고, 모든 데이터를 적어도 한 번씩 평가에 참여시켜 데이터 손실 없이 신뢰도 높은 일반화 성능을 얻으려고 씀

![train_test_split은 1회 고정 분할로 20%의 테스트 데이터가 학습에서 영구 제외되고 평가 편향이 생기지만, K-Fold는 데이터를 5등분하여 매회 다른 폴드를 테스트셋으로 삼아 5번 검증 후 평균을 내어 데이터 손실 없이 객관적인 일반화 성능을 얻는다](assets/train-test-split-vs-kfold.svg)

1. 5-Fold 분할 : 전체 데이터를 같은 크기의 5개 덩어리로 나눔
2. 회차마다 Test 자리를 한 칸 옮김 : 1회차 `[Tr, Tr, Tr, Tr, Test]` → 5회차 `[Test, Tr, Tr, Tr, Tr]`
3. 평균 성능 계산 : 5번 나온 평가 점수의 평균을 최종 성능으로 취함

```python
kfold = KFold(n_splits=5, shuffle=True, random_state=333)
scores = cross_val_score(model, x, y, cv=kfold)
print('평균 점수:', np.mean(scores))
```

- `cross_val_score`는 내부에서 `fit`과 `score`를 k번 반복하고 fold별 평가 점수 배열을 반환함
- 분류 모델(`Classifier`)은 accuracy(acc), 회귀 모델(`Regressor`)은 결정계수($R^2$)가 기본 점수로 계산됨
- 모든 데이터가 한 번씩 test 역할을 거치므로 버려지는 데이터가 없고, 특정 분할에 운 좋게 높은 점수가 나오는 것도 걸러짐
- 모델을 k번 훈련하므로 학습 시간이 단일 분할보다 k배 늘어남
- 데이터 순서에 편향이 있을 수 있으므로 나누기 전에 섞음 (`shuffle=True`)

#### cf) random_state의 착시

- 단일 분할에서 특정 `random_state`의 점수가 K-Fold 평균보다 훨씬 높게 나올 수 있음
- 모델 실력이 오른 것이 아니라 우연히 테스트셋에 맞추기 쉬운 샘플만 몰려 들어간 결과(쉬운 시험지 효과)
- 이 점수를 믿고 배포하면 실전 데이터에서 성능이 크게 떨어지므로, 요행을 피하고 안정적인 진짜 성능을 파악하려고 K-Fold를 씀

## 3. K-Fold에서의 검증 데이터

K-Fold의 각 반복(fold)에서 학습 데이터(80%)의 일부를 다시 검증(val)용으로 떼어 쓰는 방법

에포크 단위로 조기종료(EarlyStopping)를 걸거나 과적합을 감시하려면 각 fold 안에서도 검증용 데이터가 필요하기 때문

1. Fold 내부 분할 : 80% 학습 데이터 중 20%를 validation으로 뗌 (전체 데이터 기준 약 16%)
2. 검증 후 평가 : Fold 내부 train으로 가중치를 갱신하고 val로 EarlyStopping을 보며, 해당 fold의 test 데이터로 최종 점수를 기록함

- 딥러닝에서는 `fit`의 `validation_split=0.2`로 fold 내부 train에서 쉽게 떼어 쓸 수 있음
- 떼어내는 validation의 양이 일부이므로 전체적인 학습 데이터 확보에 큰 지장을 주지 않음
- fold마다 검증용 데이터도 함께 바뀌므로 특정 검증셋에 모델 튜닝이 치우치는 현상을 줄임

#### cf) Stratified K-Fold

- 분류 문제에서 각 클래스의 비율이 불균형할 때 라벨 비율을 모든 fold에 똑같이 맞춰서 나누는 K-Fold 방식
- 일반 `KFold`는 무작위로 쪼개다 특정 fold에 특정 클래스가 아예 안 들어가거나 극소수만 들어갈 수 있음
- 분류에서는 `StratifiedKFold`를 기본으로 쓰고, 회귀는 y가 연속값이므로 일반 `KFold`를 씀

## 4. 최종 가중치 결정

K-Fold 교차 검증으로 최적의 모델과 하이퍼파라미터를 찾은 뒤 실전 배포를 위한 최종 가중치를 얻는 방법

K-Fold는 공정한 평가를 위한 절차이고, 실전 배포 시에는 훈련 데이터를 최대로 활용하거나 fold별 모델을 결합해 최상의 성능을 내야 하기 때문

1. 전체 데이터 재학습 : K-Fold로 검증된 최적 설정으로, 떼어두었던 테스트 데이터까지 합쳐 전체 100% 데이터로 딱 한 번 모델을 재학습
2. K-Fold 앙상블 : k개 fold에서 학습된 k개 모델의 가중치를 모두 저장해 두고, 실전 추론 시 k개 예측값을 평균(Soft Voting) 내어 최종 결과로 사용

- 전체 재학습은 데이터 손실이 전혀 없고 모델 파일이 하나라 실무 서빙에서 가장 널리 씀
- 앙상블은 모델 k개를 모두 보관해 추론 연산이 k배 들지만, 5명의 전문가가 상의하는 구조라 단일 모델보다 예측 안정성과 점수가 가장 뛰어남